System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
import numpy as np
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [5]:
dataini = '2004-04-03 19:30:00'
datafim = '2024-04-03 19:30:00'

In [7]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
display(len(dftitulosdados))
#display(dftitulosdados.head(10))

40717

In [9]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
medM = 8  
lowM = 10
stpl = 0.02
comission = 0.0035
taxalivrerisgoprom = 0.05
drawmax = -0.1


In [11]:
#%%timeit
# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados    
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i ] = df.k.rolling(smoth).mean()
    df["d" + i ] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  
    dfstoch = df
    return dfstoch



In [13]:
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch.head(10))

,datetime,open,high,low,close,khigh,dhigh
0,2004-04-05 09:00:00,6.6778,6.6778,6.4162,6.4162,NaN,NaN
1,2004-04-05 10:00:00,6.6547,6.6770,6.5085,6.5085,NaN,NaN
2,2004-04-05 11:00:00,6.5539,6.5547,6.4393,6.5470,NaN,NaN
3,2004-04-05 12:00:00,6.5470,6.6162,6.5470,6.6085,NaN,NaN
4,2004-04-05 13:00:00,6.5701,6.5701,6.5470,6.5470,NaN,NaN
5,2004-04-05 14:00:00,6.5547,6.5547,6.5470,6.5470,NaN,NaN
6,2004-04-05 15:00:00,6.5470,6.6162,6.5470,6.5470,NaN,NaN
7,2004-04-06 09:00:00,6.5008,6.5239,6.3931,6.4778,NaN,NaN
8,2004-04-06 10:00:00,6.4162,6.4624,6.3393,6.3854,NaN,NaN
9,2004-04-06 11:00:00,6.3854,6.3854,6.2546,6.3008,NaN,NaN


In [15]:
#%%timeit
# stochastic high, med and low frcuency

def stoch_hml( dfstoch, k, d, smth, medM, lowM): 
    df = dfstoch
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k*medM, d*medM, smth*medM)
    df = stochastic(df, "low", k*medM*lowM, d*medM*lowM, smth*medM*lowM)
    dfstoch_hml = df
    return dfstoch_hml
    



In [17]:
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
display(dfstoch_hml.head(10) )

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow
0,2004-04-05 09:00:00,6.6778,6.6778,6.4162,6.4162,NaN,NaN,NaN,NaN,NaN,NaN
1,2004-04-05 10:00:00,6.6547,6.6770,6.5085,6.5085,NaN,NaN,NaN,NaN,NaN,NaN
2,2004-04-05 11:00:00,6.5539,6.5547,6.4393,6.5470,NaN,NaN,NaN,NaN,NaN,NaN
3,2004-04-05 12:00:00,6.5470,6.6162,6.5470,6.6085,NaN,NaN,NaN,NaN,NaN,NaN
4,2004-04-05 13:00:00,6.5701,6.5701,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN
5,2004-04-05 14:00:00,6.5547,6.5547,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN
6,2004-04-05 15:00:00,6.5470,6.6162,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN
7,2004-04-06 09:00:00,6.5008,6.5239,6.3931,6.4778,NaN,NaN,NaN,NaN,NaN,NaN
8,2004-04-06 10:00:00,6.4162,6.4624,6.3393,6.3854,NaN,NaN,NaN,NaN,NaN,NaN
9,2004-04-06 11:00:00,6.3854,6.3854,6.2546,6.3008,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
#%%timeit
def system_criterias (dfstoch_hml):   # input  df() =  dfstoch_hml ()
    df = dfstoch_hml
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    dfcriterias = df
    return dfcriterias


In [21]:
dfcriterias = system_criterias (dfstoch_hml)
display (dfcriterias.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh
0,2004-04-05 09:00:00,6.6778,6.6778,6.4162,6.4162,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2004-04-05 10:00:00,6.6547,6.6770,6.5085,6.5085,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2004-04-05 11:00:00,6.5539,6.5547,6.4393,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2004-04-05 12:00:00,6.5470,6.6162,6.5470,6.6085,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2004-04-05 13:00:00,6.5701,6.5701,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,2004-04-05 14:00:00,6.5547,6.5547,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6,2004-04-05 15:00:00,6.5470,6.6162,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,2004-04-06 09:00:00,6.5008,6.5239,6.3931,6.4778,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
8,2004-04-06 10:00:00,6.4162,6.4624,6.3393,6.3854,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
9,2004-04-06 11:00:00,6.3854,6.3854,6.2546,6.3008,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


dfsignals = system_signals(dfcriterias)
display (dfsignals.head(10))

In [23]:
##%%timeit
def system_signals (dfcriterias):
    df = dfcriterias
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df
    return df

In [25]:
dfsignals = system_signals (dfcriterias)
display (dfsignals.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
0,2004-04-05 09:00:00,6.6778,6.6778,6.4162,6.4162,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
1,2004-04-05 10:00:00,6.6547,6.6770,6.5085,6.5085,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
2,2004-04-05 11:00:00,6.5539,6.5547,6.4393,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
3,2004-04-05 12:00:00,6.5470,6.6162,6.5470,6.6085,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
4,2004-04-05 13:00:00,6.5701,6.5701,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
5,2004-04-05 14:00:00,6.5547,6.5547,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
6,2004-04-05 15:00:00,6.5470,6.6162,6.5470,6.5470,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
7,2004-04-06 09:00:00,6.5008,6.5239,6.3931,6.4778,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
8,2004-04-06 10:00:00,6.4162,6.4624,6.3393,6.3854,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
9,2004-04-06 11:00:00,6.3854,6.3854,6.2546,6.3008,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby


Stop Loss Reentry

In [28]:
#%%timeit
def stop_loss_reentry (dfsignals, stpl) :

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"

    return df


In [30]:
dfstoploss = stop_loss_reentry (dfsignals, stpl)
display(dfstoploss.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2006-04-05 13:00:00,5.9816,1,1,1,enter,0.119632
1,2006-04-05 14:00:00,5.9816,1,1,1,stay,0.119632
2,2006-04-05 15:00:00,5.9507,1,1,1,stay,0.088732
3,2006-04-05 16:00:00,5.9507,1,1,1,stay,0.088732
4,2006-04-06 09:00:00,5.9198,1,1,0,stay,0.057832
5,2006-04-06 10:00:00,5.8966,1,1,0,stay,0.034632
6,2006-04-06 11:00:00,5.8966,1,1,0,stay,0.034632
7,2006-04-06 12:00:00,5.8889,1,1,0,stay,0.026932
8,2006-04-06 13:00:00,5.8889,1,1,0,stay,0.026932
9,2006-04-06 14:00:00,5.9043,1,1,0,stay,0.042332


In [32]:
#%%timeit
# preparar dataframe para calcular index 
def index_dataframe (dfsignals, dfstoploss) :
    # elimino colunas de dfsignals e filtro por os valores "out"
    dfsignalsdrop = dfsignals.drop(columns=["open" ,"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    dfsignalsout = dfsignalsdrop[(dfsignalsdrop['state'] == 'out')]

    # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
    dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
    dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

    # concatenar os dois df para ter o total dos signals enter e out
    dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True)

    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignalsentoutclean = df[~cond].reset_index(drop=True)
    return dfsignalsentoutclean


In [34]:
dfsignalsentoutclean = index_dataframe (dfsignals, dfstoploss)
display (dfsignalsentoutclean)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state
0,2006-04-05 13:00:00,5.9816,1,1,1,enter
1,2006-04-06 15:00:00,5.8889,1,0,0,out
2,2006-04-20 14:00:00,5.8425,1,1,1,enter
3,2006-04-27 14:00:00,5.6802,1,1,0,out
4,2006-04-28 09:00:00,5.7421,1,1,0,enter
...,...,...,...,...,...,...
467,2024-03-15 16:00:00,21.6673,1,1,1,out
468,2024-03-18 07:00:00,22.1850,1,1,1,enter
469,2024-03-18 09:00:00,21.6812,1,1,1,out
470,2024-03-18 11:00:00,22.3236,1,1,1,enter


Index ,Trade, Index sin comission

In [36]:
#%%timeit
def index_trade(dfsignalsenteroutclean):
    df = dfsignalsentoutclean
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex


In [38]:
dfindex = index_trade(dfsignalsentoutclean)
display (dfindex.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2006-04-05 13:00:00,5.9816,1,1,1,enter,100.000000,0.000000,99.650000
1,2006-04-06 15:00:00,5.8889,1,0,0,out,98.450247,-0.015498,98.105672
2,2006-04-20 14:00:00,5.8425,1,1,1,enter,98.450247,0.000000,98.105672
3,2006-04-27 14:00:00,5.6802,1,1,0,out,95.715378,-0.027779,95.380374
4,2006-04-28 09:00:00,5.7421,1,1,0,enter,95.715378,0.000000,95.380374
5,2006-04-28 11:00:00,5.7111,1,0,1,out,95.198637,-0.005399,94.865442
6,2006-05-05 15:00:00,5.8348,1,1,1,enter,95.198637,0.000000,94.865442
7,2006-05-11 11:00:00,5.6339,1,1,0,out,91.920820,-0.034431,91.599097
8,2006-05-26 15:00:00,4.9460,1,1,1,enter,91.920820,0.000000,91.599097
9,2006-05-30 10:00:00,4.7683,1,1,0,out,88.618287,-0.035928,88.308123


stop drawdawn

In [41]:
#%%timeit
def stop_drawdawn (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < drawmax:
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdawn = df
    return dfindexdrawdawn

In [43]:
dfindexdrawdawn = stop_drawdawn(dfindex, drawmax)
display(dfindexdrawdawn)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2006-04-05 13:00:00,5.9816,1,1,1,enter,100.000000,0.000000,99.650000
1,2006-04-06 15:00:00,5.8889,1,0,0,out,98.450247,-0.015498,98.105672
2,2006-04-20 14:00:00,5.8425,1,1,1,enter,98.450247,0.000000,98.105672
3,2006-04-27 14:00:00,5.6802,1,1,0,out,95.715378,-0.027779,95.380374
4,2006-04-28 09:00:00,5.7421,1,1,0,enter,95.715378,0.000000,95.380374
...,...,...,...,...,...,...,...,...,...
467,2024-03-15 16:00:00,21.6673,1,1,1,out,133.401223,-0.016366,132.934319
468,2024-03-18 07:00:00,22.1850,1,1,1,enter,133.401223,0.000000,132.934319
469,2024-03-18 09:00:00,21.6812,1,1,1,out,130.371809,-0.022709,129.915508
470,2024-03-18 11:00:00,22.3236,1,1,1,enter,130.371809,0.000000,129.915508


Metricas

Tir anualizada Total e desvio padrão das Tir anuales Max e MIn
dataini = '2015-04-03 19:30:00'
datafim = '2019-04-03 19:30:00'


In [47]:
#%%timeit
def dataframe_tir (dfindexdrawdawn):
    dftir = dfindexdrawdawn[['datetime', 'state', 'index']]
    return dftir

In [57]:
dftir = dataframe_tir (dfindexdrawdawn)
display(dftir)

,datetime,state,index
0,2006-04-05 13:00:00,enter,99.650000
1,2006-04-06 15:00:00,out,98.105672
2,2006-04-20 14:00:00,enter,98.105672
3,2006-04-27 14:00:00,out,95.380374
4,2006-04-28 09:00:00,enter,95.380374
...,...,...,...
467,2024-03-15 16:00:00,out,132.934319
468,2024-03-18 07:00:00,enter,132.934319
469,2024-03-18 09:00:00,out,129.915508
470,2024-03-18 11:00:00,enter,129.915508


In [59]:


# Exemplo do DataFrame original
df = dftir
dataini = pd.to_datetime(dataini)
datafim = pd.to_datetime(datafim)

# Lista para novos registros
novos_registros = []

# Verifica se dataini deve ser adicionado
if dataini < df.iloc[0]['datetime']:
    novos_registros.append({
        'datetime': dataini,
        'state': '',
        'index': df.iloc[0]['index']
    })

# Verifica se datafim deve ser adicionado
if datafim > df.iloc[-1]['datetime']:
    novos_registros.append({
        'datetime': datafim,
        'state': '',
        'index': df.iloc[-1]['index']
    })

# Adiciona os registros e ordena
df = pd.concat([pd.DataFrame(novos_registros), df], ignore_index=True)
df = df.sort_values(by='datetime').reset_index(drop=True)

In [61]:
display ( df)

,datetime,state,index
0,2004-04-03 19:30:00,,99.650000
1,2006-04-05 13:00:00,enter,99.650000
2,2006-04-06 15:00:00,out,98.105672
3,2006-04-20 14:00:00,enter,98.105672
4,2006-04-27 14:00:00,out,95.380374
...,...,...,...
469,2024-03-18 07:00:00,enter,132.934319
470,2024-03-18 09:00:00,out,129.915508
471,2024-03-18 11:00:00,enter,129.915508
472,2024-03-26 14:00:00,out,139.006963


In [63]:


# Determina os anos, excluindo o último ano
ano_inicial = df['datetime'].min().year
ano_final = (df['datetime'].max().year)

# Gera os anos do intervalo EXCLUINDO o último ano
anos_validos = range(ano_inicial, ano_final-1 )  # << ajuste aqui

# Lista para os novos registros
novos_registros = []

for ano in anos_validos:
    fim_do_ano = pd.to_datetime(f'{ano}-12-31 23:59:59')
    df_antes = df[df['datetime'] < fim_do_ano]
    if not df_antes.empty:
        index_valor = df_antes.iloc[-1]['index']
        novos_registros.append({
            'datetime': fim_do_ano,
            'state': '',
            'index': index_valor
        })

# Adiciona e organiza
df = pd.concat([df, pd.DataFrame(novos_registros)], ignore_index=True)
df = df.sort_values('datetime').reset_index(drop=True)

In [70]:
#pd.set_option('display.max_rows', None)
display (ano_final)
display (anos_validos)
display (df)



2024

range(2004, 2023)

,datetime,state,index
0,2004-04-03 19:30:00,,99.650000
1,2004-12-31 23:59:59,,99.650000
2,2005-12-31 23:59:59,,99.650000
3,2006-04-05 13:00:00,enter,99.650000
4,2006-04-06 15:00:00,out,98.105672
5,2006-04-20 14:00:00,enter,98.105672
6,2006-04-27 14:00:00,out,95.380374
7,2006-04-28 09:00:00,enter,95.380374
8,2006-04-28 11:00:00,out,94.865442
9,2006-05-05 15:00:00,enter,94.865442


In [72]:
import pandas as pd
import numpy as np

# Suponha que df seja seu DataFrame original com colunas: 'datetime', 'state', 'index'
df['datetime'] = pd.to_datetime(df['datetime'])

# ✅ Passo 1: consolidar por dia e manter o último registro
df['date'] = df['datetime'].dt.date
df_diario = df.sort_values('datetime').groupby('date', as_index=False).last()

# ✅ Passo 2: selecionar datas de fim de ano
df_fim_ano = df_diario[
    (pd.to_datetime(df_diario['date']).dt.month == 12) &
    (pd.to_datetime(df_diario['date']).dt.day == 31)
].copy()

# ✅ Passo 3: calcular TIR entre pares de fim de ano
resultados = []

for i in range(1, len(df_fim_ano)):
    dt_inicio = pd.to_datetime(df_fim_ano.iloc[i - 1]['date'])
    dt_fim = pd.to_datetime(df_fim_ano.iloc[i]['date'])
    idx_inicio = df_fim_ano.iloc[i - 1]['index']
    idx_fim = df_fim_ano.iloc[i]['index']
    dias = (dt_fim - dt_inicio).days

    if dias > 0 and idx_inicio != 0:
        tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
        resultados.append({
            'datetime': dt_fim,
            'tiranual': tir
        })

# ✅ Passo 4: adicionar último intervalo incompleto
if not df_fim_ano.empty:
    dt_inicio = pd.to_datetime(df_fim_ano.iloc[-1]['date'])
    idx_inicio = df_fim_ano.iloc[-1]['index']
    dt_fim = pd.to_datetime(df_diario.iloc[-1]['date'])
    idx_fim = df_diario.iloc[-1]['index']
    dias = (dt_fim - dt_inicio).days

    if dias > 0 and idx_inicio != 0:
        tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
        resultados.append({
            'datetime': dt_fim,
            'tiranual': tir
        })

# ✅ Passo 5: criar DataFrame final
dftiranual = pd.DataFrame(resultados)

In [74]:
display (dftiranual)

,datetime,tiranual
0,2005-12-31,0.000000
1,2006-12-31,0.074673
2,2007-12-31,0.055653
3,2008-12-31,-0.131627
4,2009-12-31,-0.283463
5,2010-12-31,0.141655
6,2011-12-31,-0.086982
7,2012-12-31,0.251925
8,2013-12-31,0.447476
9,2014-12-31,-0.132938


In [82]:
# Supondo que dftiranual tenha uma coluna 'tiranual'

tir_max = dftiranual['tiranual'].max()
tir_min = dftiranual['tiranual'].min()
tir_std = dftiranual['tiranual'].std()

# Exibindo os resultados formatados

print(f"🔼 Máximo da TIR: {tir_max:.6%}")
print(f"🔽 Mínimo da TIR: {tir_min:.6%}")
print(f"📐 Desvio padrão da TIR: {tir_std:.6%}")

🔼 Máximo da TIR: 44.747637%
🔽 Mínimo da TIR: -28.346256%
📐 Desvio padrão da TIR: 21.128108%


In [88]:
def calcular_tir_total_anualizada(df):
    # Garante que datetime está no formato certo
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtra enter e out
    df_enter = df[df['state'] == 'enter']
    df_out = df[df['state'] == 'out']

    # Verificação
    if df_enter.empty or df_out.empty:
        return None

    # Índice inicial e final
    idx_inicio = df_enter.iloc[0]['index']
    idx_fim = df_out.iloc[-1]['index']

    # Período completo entre primeira e última data do DataFrame
    dt_inicio_total = df['datetime'].min()
    dt_fim_total = df['datetime'].max()
    dias_total = (dt_fim_total - dt_inicio_total).days

    # Validação
    if dias_total <= 0 or idx_inicio == 0:
        return None

    # TIR anualizada com base no período total do df
    tir_anual = (idx_fim / idx_inicio) ** (365 / dias_total) - 1

    return {
        'data_inicio_real': dt_inicio_total,
        'data_fim_real': dt_fim_total,
        'index_inicio': idx_inicio,
        'index_fim': idx_fim,
        'dias_total': dias_total,
        'tir_anualizada': tir_anual
    }

In [90]:
resultado = calcular_tir_total_anualizada(df)

if resultado:
    print(f"📍 Período total do DataFrame: {resultado['data_inicio_real']} → {resultado['data_fim_real']}")
    print(f"📈 Índice inicial (enter): {resultado['index_inicio']:.6f}")
    print(f"📈 Índice final   (out)  : {resultado['index_fim']:.6f}")
    print(f"📅 Dias no período total: {resultado['dias_total']}")
    print(f"📊 TIR anualizada (base no período total): {resultado['tir_anualizada']:.6%}")
else:
    print("⚠️ Dados insuficientes para calcular a TIR.")

📍 Período total do DataFrame: 2004-04-03 19:30:00 → 2024-04-03 19:30:00
📈 Índice inicial (enter): 99.650000
📈 Índice final   (out)  : 139.006963
📅 Dias no período total: 7305
📊 TIR anualizada (base no período total): 1.677068%


In [77]:
def datas_drawdown_max(df):
    df = df.copy()
    df = df[df["index"].notna()]
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Série com índice acumulado
    acumulado = df["index"]
    pico = acumulado.cummax()
    drawdown = acumulado - pico

    # Índice do drawdown máximo
    idx_vale = drawdown.idxmin()
    idx_pico = (acumulado[:idx_vale]).idxmax()
    idx_trademin = df["trade"].idxmin()


    # Datas correspondentes
    data_pico = df.loc[idx_pico, "datetime"]
    data_vale = df.loc[idx_vale, "datetime"]
    data_trademin = df.loc[idx_trademin, "datetime"]

    # Diferença percentual
    valor_pico = df.loc[idx_pico, "index"]
    valor_vale = df.loc[idx_vale, "index"]
    drawdown_pct = ((valor_vale - valor_pico) / valor_pico) * 100
    trademin = df["trade"].min()
    
    return pd.Series({
        "Data do Pico": data_pico.strftime("%Y-%m-%d %H:%M"),
        "Data do Vale": data_vale.strftime("%Y-%m-%d %H:%M"),
        "Valor do Pico": round(valor_pico, 2),
        "Valor do Vale": round(valor_vale, 2),
        "Drawdown Máximo (%)": f"{drawdown_pct:.2f}%",
        "Màxima perdida por trade": f"{df["trade"].min()*100 :.2f}%",
        "Data do Max Loser Trade": data_trademin.strftime("%Y-%m-%d %H:%M"),
    })

In [79]:
resultado_datas = datas_drawdown_max(df)
print(resultado_datas)

Data do Pico                2019-07-01 14:00
Data do Vale                2022-05-12 09:00
Valor do Pico                          102.0
Valor do Vale                          57.41
Drawdown Máximo (%)                  -43.72%
Màxima perdida por trade              -5.23%
Data do Max Loser Trade     2021-10-07 08:00
dtype: object


In [173]:
def estatisticas_trades(df):
    df = df.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) * 100

    resultado = {
        "Total de Trades": len(trades),
        "Percentual de Trades Positivos (%)": f"{porcentagem_pos:.2f}%",
        "Média dos Trades Positivos": round(positivos.mean(), 6),
        "Desvio Padrão (Trades Positivos)": round(positivos.std(), 6),
        "Média dos Trades Negativos": round(negativos.mean(), 6),
        "Desvio Padrão (Trades Negativos)": round(negativos.std(), 6)
    }

    return pd.Series(resultado)

In [175]:
resumo_trades = estatisticas_trades(df)
print(resumo_trades)

Total de Trades                             60
Percentual de Trades Positivos (%)      16.67%
Média dos Trades Positivos            0.048725
Desvio Padrão (Trades Positivos)      0.044806
Média dos Trades Negativos           -0.016233
Desvio Padrão (Trades Negativos)      0.013479
dtype: object


In [85]:
def ranking_drawdowns_puros(df, coluna="index", top_n=5):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").head(top_n).reset_index(drop=True)

In [87]:
ranking_drawdowns_puros(df, coluna="index", top_n=5)


,Data Pico,Valor Pico,Data Vale,Valor Vale,Drawdown (%)
0,2019-07-01 14:00:00,102.004024,2022-05-12 09:00:00,57.409405,-43.72
1,2019-04-03 09:00:00,99.650000,2019-04-04 11:00:00,96.305464,-3.36
2,2019-04-15 14:00:00,100.271202,2019-06-26 13:00:00,97.024371,-3.24


In [89]:
def estatisticas_drawdowns_puros(df, coluna="index"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            if vale_idx is not None and max_dd < 0:
                drawdowns.append(max_dd * 100)  # salva como porcentagem
            # novo ciclo
            pico = serie[i]
            pico_idx = i
            max_dd = 0
            vale_idx = None
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # salva último ciclo, se houver
    if vale_idx is not None and max_dd < 0:
        drawdowns.append(max_dd * 100)

    # série com drawdowns reais
    serie_dd = pd.Series(drawdowns)

    estatisticas = {
        "Total de Drawdowns": len(drawdowns),
        "Média dos Drawdowns (%)": round(serie_dd.mean(), 2),
        "Desvio Padrão (%)": round(serie_dd.std(), 2),
        "Drawdown Máximo (%)": round(serie_dd.min(), 2),
        "Drawdown Mínimo (%)": round(serie_dd.max(), 2)
    }

    return pd.Series(estatisticas)

In [91]:
resumo_dd_puros = estatisticas_drawdowns_puros(df)
print(resumo_dd_puros)

Total de Drawdowns          3.00
Média dos Drawdowns (%)   -16.77
Desvio Padrão (%)          23.34
Drawdown Máximo (%)       -43.72
Drawdown Mínimo (%)        -3.24
dtype: float64


In [177]:
def estatistica_periodos_estaticos_em_dias(df, coluna="index_sc"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({
                "Valor index": grupo[coluna].iloc[0],
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "Duração (dias)": duracao_dias,
                "Número de Registros": len(grupo)
            })

    df_resultado = pd.DataFrame(periodos_estaticos)

    if df_resultado.empty:
        resumo = {
            "Total de Períodos Estáticos": 0,
            "Duração Média (dias)": 0,
            "Maior Duração (dias)": 0,
            "Data do Maior Período": {"Data Início": None, "Data Fim": None}
        }
    else:
        resumo = {
            "Total de Períodos Estáticos": len(df_resultado),
            "Duração Média (dias)": round(df_resultado["Duração (dias)"].mean(), 2),
            "Maior Duração (dias)": df_resultado["Duração (dias)"].max(),
            "Data do Maior Período": df_resultado.loc[df_resultado["Duração (dias)"].idxmax(), ["Data Início", "Data Fim"]].to_dict()
        }

    return df_resultado, pd.Series(resumo)

In [179]:
df_periodos, estatisticas = estatistica_periodos_estaticos_em_dias(df)
display(df_periodos)
display(estatisticas)

,Valor index,Data Início,Data Fim,Duração (dias),Número de Registros
0,101.833293,2016-04-25 11:00:00,2016-07-14 16:00:00,80,2
1,100.186849,2016-07-20 13:00:00,2016-08-09 14:00:00,20,2
2,99.646697,2016-08-11 11:00:00,2016-11-07 15:00:00,88,2
3,96.893201,2016-11-09 15:00:00,2017-02-07 12:00:00,89,2
4,101.054558,2017-02-16 15:00:00,2017-03-07 16:00:00,19,2
5,116.059841,2017-03-28 10:00:00,2017-04-06 13:00:00,9,2
6,114.084405,2017-04-10 13:00:00,2017-05-03 09:00:00,22,2
7,120.338571,2017-05-12 11:00:00,2017-05-31 13:00:00,19,2
8,122.288214,2017-06-07 11:00:00,2017-09-28 10:00:00,112,2
9,129.049297,2017-10-11 12:00:00,2017-10-17 14:00:00,6,2


Total de Períodos Estáticos                                                   29
Duração Média (dias)                                                       32.38
Maior Duração (dias)                                                         329
Data do Maior Período          {'Data Início': 2018-02-13 14:00:00, 'Data Fim...
dtype: object